Model Optimization & Hyperparameter Tuning

In [12]:
# Step1 : setup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# if xgboost is installed
try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False

# load cleaned dataset
df = pd.read_csv('E:/CY Tech/Big Data project/Project 1 Car Price Prediction Multiple Linear Regression/Data/Processed/car_price_clean.csv')

# separate target and features
y = df['price']
X = df.drop('price', axis=1)

# identify column types
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_test_processed.shape

# load baseline results
results_df = pd.read_csv('E:/CY Tech/Big Data project/Project 1 Car Price Prediction Multiple Linear Regression/Data/Processed/baseline_results.csv')

1- Objective:

- Improve model performance using hyperparameter tuning and cross-validation (k=5).
- Compare the tuned results with the baseline models.

In [13]:
# Step 2 : Define modelss and parametere grids

models_to_tune = {
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor()
}

# add xgboost if available
if xgb_available:
    models_to_tune["XGBoost"] = XGBRegressor(random_state=42, objective='reg:squarederror')

# parameter grids for each model
param_grids = {
    "RandomForest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5]
    },

    "GradientBoosting": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3]
    },

    "SVR": {
        "kernel": ["rbf", "linear"],
        "C": [1, 5, 10],
        "gamma": ["scale"]
    },

    "KNN": {
        "n_neighbors": [3, 5, 7],
        "weights": ["uniform", "distance"]
    }
}

# include xgboost params if available
if xgb_available:
    param_grids["XGBoost"] = {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5]
    }

2- Models to Optimize (with GridSearchCV):

- Random Forest Regressor
- Gradient Boosting Regressor
- XGBoost Regressor
- Support Vector Regressor (SVR)
- K-Nearest Neighbors Regressor

In [14]:
# Step 3: Hyperparameter tuning with GridSearchCV

best_models = {}
tuning_results = []

for name, model in models_to_tune.items():
    print("Tuning model:", name)

    param_grid = param_grids[name]

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X_train_processed, y_train)

    best_models[name] = grid.best_estimator_

    tuning_results.append({
        "Model": name,
        "Best Params": grid.best_params_,
        "Best Score (neg MSE)": grid.best_score_
    })

    print("  Best params:", grid.best_params_)
    print("  Best neg MSE:", grid.best_score_)
    print("")

print("Tuning completed.")

Tuning model: RandomForest
  Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
  Best neg MSE: -6067424.780521875

Tuning model: GradientBoosting
  Best params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 200}
  Best neg MSE: -5638271.495681344

Tuning model: SVR
  Best params: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
  Best neg MSE: -25384293.336136587

Tuning model: KNN
  Best params: {'n_neighbors': 3, 'weights': 'distance'}
  Best neg MSE: -10848580.956275817

Tuning completed.


3- Hyperparameter Tuning Approach:

- Use GridSearchCV with 5-fold cross-validation.
- Optimize key parameters for each model (e.g., depth, estimators, learning rate, kernel).
- Track best parameters and best scores.

In [15]:
# Step4: Evaluate tuned models on the test set

optimized_results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test_processed)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    optimized_results.append({
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R²": round(r2, 4)
    })

optimized_df = pd.DataFrame(optimized_results)
optimized_df

,Model,MAE,RMSE,R²
0,RandomForest,1354.08,1895.05,0.9545
1,GradientBoosting,1822.26,2360.91,0.9294
2,SVR,3066.88,5593.03,0.6037
3,KNN,2036.45,3244.23,0.8667


4- Performance Evaluation:

For each optimized model, calculate:

- MAE
- RMSE
- R² Score
- Best parameter set from GridSearchCV

In [5]:
#best hyperparameters for each optimized model
best_params_list = []

for name, model in best_models.items():
    params = model.get_params()
    best_params_list.append({
        "Model": name,
        "Best Params": params
    })

best_params_df = pd.DataFrame(best_params_list)
best_params_df

,Model,Best Params
0,RandomForest,"{'bootstrap': True, 'ccp_alpha': 0.0, 'criteri..."
1,GradientBoosting,"{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': ..."
2,SVR,"{'C': 10, 'cache_size': 200, 'coef0': 0.0, 'de..."
3,KNN,"{'algorithm': 'auto', 'leaf_size': 30, 'metric..."


5- Comparison: Baseline vs Optimized:

- Create a summary table comparing the best baseline and optimized models.
- Identify which tuning produced the best improvement and why.

In [16]:
# Step 5: Comparison: Baseline vs Optimized

# load baseline results
results_df = pd.read_csv('E:/CY Tech/Big Data project/Project 1 Car Price Prediction Multiple Linear Regression/Data/Processed/baseline_results.csv')

# sort both tables by R²
baseline_sorted = results_df.sort_values(by='R²', ascending=False).reset_index(drop=True)
optimized_sorted = optimized_df.sort_values(by='R²', ascending=False).reset_index(drop=True)

print("Baseline models (sorted):")
display(baseline_sorted)

print("Optimized models (sorted):")
display(optimized_sorted)

Baseline models (sorted):


,Model,MAE,RMSE,R²
0,Random Forest,1380.47,1938.93,0.9524
1,Ridge Regression,1823.77,2762.44,0.9033
2,Lasso Regression,1913.83,2845.79,0.8974
3,Decision Tree,2098.49,3427.12,0.8512
4,Linear Regression,3152.93,4961.85,0.6881


Optimized models (sorted):


,Model,MAE,RMSE,R²
0,RandomForest,1354.08,1895.05,0.9545
1,GradientBoosting,1822.26,2360.91,0.9294
2,KNN,2036.45,3244.23,0.8667
3,SVR,3066.88,5593.03,0.6037


6- Final Model Selection:

- Select the top-performing tuned model for final analysis and future deployment.
- Save the trained model for later use (optional app or API integration).

In [19]:
# Step 6 : Final Model Selection

# get best optimized model based on R²
best_row = optimized_df.sort_values(by='R²', ascending=False).iloc[0]
best_model_name = best_row['Model']
best_model = best_models[best_model_name]

print("Best optimized model:", best_model_name)
print("R²:", best_row['R²'])

# optional: save the model
from joblib import dump
dump(best_model, 'E:/CY Tech/Big Data project/Project 1 Car Price Prediction Multiple Linear Regression/Models/final_regression_model.joblib')

Best optimized model: RandomForest
R²: 0.9545


['E:/CY Tech/Big Data project/Project 1 Car Price Prediction Multiple Linear Regression/Models/final_regression_model.joblib']

*Optimized Conclusion*

- The best optimized model was the Random Forest.
- It achieved a tuned R² of 0.9545, which is an improvement compared to the best baseline R² of 0.9524.
- The improvement comes from adjusting key hyperparameters during tuning, which allowed the model to fit the data patterns more accurately without overfitting.